In [1]:
import os
import json
from typing import List, Literal, Optional, Text, Union, Dict, Any
from IPython.display import Image, display
from langchain_core.runnables.graph_mermaid import MermaidDrawMethod
from agents.utilities.agent_utils import save_result_to_json
from agents.llm_model import UnifiedModel, model_name
from load_data import extract_modified_triplesets_from_file
from agents.agent_prompts import (END_TO_END_GENERATION_PROMPT_EN, 
                                  END_TO_END_GENERATION_PROMPT_GA, 
                                  input_prompt,)
from agents.agents_modules.workflow import (build_agent_workflow,
                                            build_agent_workflow_single_module,
                                            build_agent_workflow_no_guardrail,
                                            build_agent_workflow_no_finalizer,
                                            build_agent_workflow_no_orchestrator,)

/home/chinonso/anaconda3/envs/lang2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-12-01 11:18:52.923871: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-01 11:18:52.951580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-01 11:18:53.489953: I tensorflow/core/util/port.cc:153] oneDNN custom operatio

In [ ]:
provider = "openai" #ollama, openai, hf, aixplain

# Default Architecture
process_flow = build_agent_workflow(provider=provider)

# Ablation Study
graph_single = build_agent_workflow_single_module(provider="ollama", language="en") # Single module for CO, TS & SR
graph_no_guardrail = build_agent_workflow_no_guardrail(provider="ollama", language="ga") # No guardrail
graph_no_finalizer = build_agent_workflow_no_finalizer(provider="ollama", language="ga") # No finalizer
graph_no_orchestrator = build_agent_workflow_no_orchestrator(provider="ollama", language="ga") # No Orchestrator

architecture = [process_flow, graph_single, graph_no_guardrail, graph_no_finalizer, graph_no_orchestrator]

# display(Image(architecture[0].get_graph(xray=True).draw_mermaid_png()))

In [3]:
triplesets = extract_modified_triplesets_from_file("data/D2T-1-FA_same3to6_min50_max500_sample.xml")
triple_length = [{i : len(j)} for i, j in enumerate(triplesets, start=1)]
# triple_length = {i : len(j) for i, j in enumerate(triplesets, start=1)}

print(f"Extracted {len(triplesets)} modified triplesets.")
print(f"Triple lengths: {triple_length}")

Extracted 10 modified triplesets.
Triple lengths: [{1: 156}, {2: 252}, {3: 149}, {4: 130}, {5: 174}, {6: 209}, {7: 201}, {8: 219}, {9: 204}, {10: 140}]


In [4]:
num = 2
data = triplesets[num]
# data

In [ ]:
query = f"""You are an agent designed to generate text from data for a data-to-text natural language generation. You can be provided data in the form of xml, table, meaning representations, graphs etc. 
Your task is to generate the appropriate text given the data information without omitting any field or adding extra information in essence called hallucination.

Dataset: 'webnlg'
Here is the data, generate text using the provided data:
{data}"""

initial_state = {
    "data_input": data,
    "user_prompt": query,
    "raw_data": data,
    "history_of_steps": [],
    "final_response": "",
    "next_agent": "",
    "next_agent_payload": "",
    "current_step": 0,
    "iteration_count": 0,
    "max_iteration": 100,
}

state = architecture[0].invoke(initial_state, config={"recursion_limit": initial_state["max_iteration"]})
prediction = state['final_response']



> Entering new AgentExecutor chain...
Action:
```
{
  "action": "Final Answer",
  "action_input": [
    ["Bucharest", "description", "Capital den largest city for Romania"],
    ["Bucharest", "description", "Haaptstad vu Rumänien"],
    ["Bucharest", "description", "Hauptstadt Rumäniens"],
    ["Bucharest", "description", "Kryeqyteti i Rumanisë"],
    ["Bucharest", "description", "thủ đô và là thành phố lớn nhất của Romania"],
    ["Bucharest", "country", "Romania"],
    ["Bucharest", "type", "București_-_Ilfov"],
    ["Bucharest", "subdivision", "Development_regions_of_Romania"],
    ["Ilfov_County", "subdivision", "Bucharest"],
    ["Muntenia", "subdivision", "Bucharest"],
    ["Bucharest", "areaTotal", "240.0"],
    ["Bucharest", "areaMetro", "1803.0"],
    ["Bucharest", "elevation", "55.8"],
    ["Bucharest", "populationTotal", "1716961"],
    ["Bucharest", "populationTotalRanking", "1"],
    ["Bucharest", "populationDensity", "7196.0"],
    ["Bucharest", "populationMetro", "2303

KeyboardInterrupt: 

In [ ]:
# save_result_to_json(state, dataset_folder=f"{name}", filename=f"{name}_{num}.json")
save_result_to_json(state, filename=f"new_{num}.json")
print(prediction)

[SAVED] Agent result saved to: results/new_2.json
Bucharest is the capital and largest city of Romania, known in various languages as 'Haaptstad vu Rumänien,' 'Hauptstadt Rumäniens,' 'Kryeqyteti i Rumanisë,' and 'thủ đô và là thành phố lớn nhất của Romania.' The city is located in Romania and carries the motto 'The Homeland and my right.' It is classified as part of the București-Ilfov region.

Covering a total area of 240.0 square kilometers, Bucharest's metropolitan area extends to 1,803.0 square kilometers. The city sits at an elevation of 55.8 meters and is part of the Development Regions of Romania. It is also associated with Ilfov County and the historical region of Muntenia. Lake Dâmbovița is the nearest significant body of water to the city.

Bucharest has a population of 1,716,961, making it the most populous city in Romania. The metropolitan area is home to 2,303,505 people. The city has a population density of 7,196 inhabitants per square kilometer, while the metropolitan ar

In [ ]:
data

[['Bucharest', 'areaMetro', '1803.0'],
 ['Bucharest', 'areaTotal', '240.0'],
 ['Bucharest', 'populationDensity', '7196.0'],
 ['Bucharest', 'populationMetroDensity', '1277.0'],
 ['Bucharest', 'areaCode', '+40 31'],
 ['Bucharest', 'country', 'Romania'],
 ['Bucharest', 'description', 'Capital den largest city for Romania'],
 ['Bucharest', 'description', 'Haaptstad vu Rumänien'],
 ['Bucharest', 'description', 'Hauptstadt Rumäniens'],
 ['Bucharest', 'description', 'Kryeqyteti i Rumanisë'],
 ['Bucharest', 'description', 'thủ đô và là thành phố lớn nhất của Romania'],
 ['Bucharest', 'elevation', '55.8'],
 ['Bucharest', 'founder', 'Vlad_the_Impaler'],
 ['Bucharest', 'governmentType', 'Mayor–council_government'],
 ['Bucharest', 'motto', "('The Homeland and my right')"],
 ['Bucharest', 'populationMetro', '2303505'],
 ['Bucharest', 'populationTotal', '1716961'],
 ['Bucharest', 'populationTotalRanking', '1'],
 ['Bucharest', 'postalCode', '0100xx-0201xx, 0201xx-0300xx, 0365xx'],
 ['Bucharest', 'sub

In [ ]:
bucharest = """Bucharest, located in Romania, is classified as "București - Ilfov" and serves as the country's capital and largest city. The city operates under a mayor–council government system and is known by the motto "The Homeland and my right." Bucharest's area code is +40 31, and its postal codes range from 0100xx-0201xx, 0201xx-0300xx, and 0365xx. The city observes both Eastern European Time and Eastern European Summer Time, with UTC offsets of +02:00 and +03:00, respectively. Bucharest was founded by Vlad the Impaler. It is described in various languages as the capital and largest city of Romania, including "Capital den largest city for Romania," "Haaptstad vu Rumänien," "Hauptstadt Rumäniens," "Kryeqyteti i Rumanisë," and "thủ đô và là thành phố lớn nhất của Romania."

Geographically, Bucharest covers a total area of 240.0 square kilometers, with a metropolitan area spanning 1,803.0 square kilometers. The city is situated at an elevation of 55.8 meters and is part of the Development Regions of Romania. Additionally, Ilfov County and Muntenia are subdivisions associated with Bucharest.

In terms of demographics, Bucharest has a total population of 1,716,961, while its metropolitan area is home to 2,303,505 people. The population density within the city is 7,196.0 inhabitants per square kilometer, and the metropolitan density is 1,277.0. Bucharest ranks first in population among Romanian cities. The city is home to diverse communities, including Turks of Romania, Germans of Romania, and Romani people in Romania.

Bucharest has played a significant historical and political role, serving as the capital of the Socialist Republic of Romania, Wallachia, and the Kingdom of Romania. It was also the largest city in both the Kingdom of Romania and the Kingdom of Romania under Fascism.

Several important historical events have taken place in Bucharest, such as the Wallachian Revolution of 1848, the Raids on Bucharest, the Battle of Bucharest, and the Bombing of Bucharest during World War II.

Many notable individuals were born in Bucharest, including Cristian Nemescu, Arthur Caesar, George Ogăraru, and Lazăr Edeleanu. The city is also the place of death for figures such as Cristian Nemescu, Sergiu Nicolaescu, Lazăr Edeleanu, and Nicolae Golescu. Residents have included Petre Cișmigu, Alexandra Cadanțu-Ignatik, and Irina-Camelia Begu, while the resting places of Nicolae Tonitza, Augustin Buzura, Alecu Filipescu-Vulpea, and Petre Ispirescu are found in the city.

Bucharest is associated with the education and training of many individuals, including Zoe Țapu, Florin-Alexandru Alexe, Laurențiu Rebega, Alex Gâlmeanu, Alina Eremia, Daniel Ioniță (poet), Jacques Hérold, Roberto Dutesco, and Marilena Murariu. It is also home to institutions and employers such as Erwin Friedlander, Irinel Popescu, Ștefan S. Nicolau, and Mircea Dumitrescu.

Religiously, Bucharest is the diocese for Patriarch Nicodim of Romania, Mihai Frățilă, Patriarch Daniel of Romania, Teoctist Arăpașu, and Patriarch Iustin of Romania. The Roman Catholic Archdiocese of Bucharest is also based here. The city is the place of canonization for Dionysius Exiguus, Constantin Brâncoveanu (along with Constantin, Ștefan, Radu, and Matei Brâncoveanu), Stephen the Great, and Neagoe Basarab. Major shrines in Bucharest include those dedicated to Constantin Brâncoveanu and Vladimir Ghika, who was also beatified in the city.

Economically, Bucharest hosts the locations of BRD – Groupe Société Générale, First Bank (Romania), Mega Image, Fun Labs, and Societatea Pentru Exploatări Tehnice. The headquarters of Dan Air (Romania), the Ministry of Internal Affairs (Romania), the Social Liberal Union, and the Social Liberal Humanist Party are all in Bucharest. The Foreign Ministry of the Independent State of Croatia has a child organization in the city.

Bucharest is a center for education and research, being home to the Military Technical Academy, the University of Agronomic Sciences and Veterinary Medicine of Bucharest, and the college attended by Constantin Herold.

Culturally, the city is served by Radio România Muzical, Virgin Radio Romania, Dance FM, and the Bucharest Bar. Songs such as "Tornerò" by Mihai Trăistariu, "Vanilla Chocolat," and "Cherry Pop" were recorded in Bucharest. The city is also associated with the Joffre cake and Silviu Prigoană.

In sports, Bucharest is the club or ground for Gheorghiță Ștefan, Daniela Druncea, CS Smart Sport Bucharest, and Faur București. It is also the position for ASC Daco-Getica București, FC Carmen București, and FC Venus București, and was the host city for the 1993 Cupa României final.

The city's infrastructure and transportation network includes the Centura București and Bucharest Ring Motorway beltways, the Basarab Overpass, and the starting points for the A1, A2, and A3 motorways, as well as the Bucharest–Giurgiu Motorway and DN6. The A3 motorway also has a junction in Bucharest. The Constanța railway station serves railway lines to the city, and CFRNA targets Bucharest's airport. Romaero, Stadionul Național (1953), and FCSB Arena Națională are located in Bucharest, and Lake Dâmbovița is the nearest city to the lake.

Bucharest is involved in industry and manufacturing, being the manufacturer of A Vlaicu II and the assembly location for the Lada Kalina (first generation), Lada Priora, and Lada Niva.

The city is also a military center, serving as the garrison for the Romanian Naval Forces, Romanian Military Police, Romanian Land Forces, and the Royal Romanian Air Force.

Architecturally, Bucharest features significant buildings associated with Carol Benesch, Haralamb H. Georgescu, and Toma N. Socolescu.

Other notable aspects of Bucharest include the Sala Palatului, and it is the hometown of Mandinga (band), Kamara Ghedi, ConTempo Quartet, and Elena Gheorghe. The city is also the constituency district for Crin Antonescu (Tenure 6 and 7) and Nicolae Fleva (Tenure 2).
"""
# print(bucharest)

In [ ]:
# %pip install -U "openai>=1.60.0" \
#               "langchain>=0.3.19" \
#               "langchain-openai>=0.3.33"\
# %pip install -U openai langchain langchain-openai
# %pip install -U "langchain-classic>=0.1.0"
# %pip install tf-keras